- Chemical multiverse visualization.
- Environment preparation.

In [1]:
import warnings
warnings.filterwarnings("ignore")

- Load modules.

In [2]:
from common_funtions import *

In [3]:
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, MACCSkeys
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from rdkit.Chem import rdFingerprintGenerator
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import molplotly
import kaleido
from plotly import io as pio
from dash import Dash
from transformers import AutoModelForMaskedLM, AutoTokenizer

- Import data.

In [4]:
df = pd.read_csv("../raw_data/curated_PrimaryOdor_1.csv")
df.columns

Index(['standardized_smiles', 'CAS-Id', 'fcfp_compact', 'fcfp_environments',
       'odor_1', 'odor_2', 'odor_3', 'odor_4', 'odor_5', 'odor_6', 'odor_7',
       'odor_8', 'odor_9', 'odor_10', 'odor_11', 'odor_12', 'odor_13',
       'odor_14', 'odor_15', 'odor_16', 'odor_17', 'odor_18', 'odor_19',
       'odor_20', 'odor_21', 'odor_22', 'odor_23', 'odor_24', 'odor_25',
       'odor_26', 'odor_27', 'odor_28', 'odor_29', 'odor_30', 'odor_31',
       'odor_32', 'odor_33', 'odor_34', 'odor_35', 'odor_36'],
      dtype='object')

In [5]:
df.drop(columns = ["fcfp_compact", "fcfp_environments"], inplace = True)
cols = ["CAS-Id"] + [col for col in df.columns if col != "CAS-Id"]
df = df[cols]
df.columns = df.columns.str.lower().str.replace("-", "_")
df.head(2)


,cas_id,standardized_smiles,odor_1,odor_2,odor_3,odor_4,odor_5,odor_6,odor_7,odor_8,...,odor_27,odor_28,odor_29,odor_30,odor_31,odor_32,odor_33,odor_34,odor_35,odor_36
0,103-64-0,BrC=Cc1ccccc1,Green,Fruity,Vegetation,Fragrant,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,29171-20-8,C#CC(C)(O)CCC=C(C)C,Sweet,Plants,Waxy,Ambrosial,Resinous,Grassy,Dry,Woody,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Define the odor columns
odor_columns = [f"odor_{i}" for i in range(1, 37)]

# Apply lowercase and space-to-underscore replacement
for col in odor_columns:
    df[col] = df[col].astype(str).str.lower().str.replace(" ", "_")

In [7]:
df.head(2)

,cas_id,standardized_smiles,odor_1,odor_2,odor_3,odor_4,odor_5,odor_6,odor_7,odor_8,...,odor_27,odor_28,odor_29,odor_30,odor_31,odor_32,odor_33,odor_34,odor_35,odor_36
0,103-64-0,BrC=Cc1ccccc1,green,fruity,vegetation,fragrant,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
1,29171-20-8,C#CC(C)(O)CCC=C(C)C,sweet,plants,waxy,ambrosial,resinous,grassy,dry,woody,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan


In [8]:
final_cluster_assignment = {
    "Waste": "septic, cleaning_materials, fecal, industrial_odours, microbiological, microbial, waste, pungent, musty",
    "Food": "restaurant, boiled, food, baked, cabbage, cooked, vegetables, chocolate, non-vegetarian, popcorn, non-food_items, grains, dairy, fishy, bakery, fried",
    "Woody": "woody, oriental, ylang, ozonou, woody_oriental, ambrosial, dry_woods, soft_oriental, fragrant, mossy_woods, soft_floral, pine, balsamic, spices",
    "Decomposition": "unripe, sulphur, sulp, rancid, sewery, chlorinous, hircine, alliaceous, dead_animal, putrid, brown, foul",
    "Sweet": "marshy, fruity, smokey, minty, sweet, earthy, lem, nutty, fresh, clean, caramel",
    "Citrus": "green, ripe, lemon, non-citrus_fruity, dry_fruit, citrus",
    "Bodily": "bland, bitter, sharp, burnt, waxy, bloody, salty, nauseating, sickening, mol, dry, raw, metallic, people_&_animals, tobacco_smoke, poultry",
    "Synthetic": "solvent, synthetic, resinous, terpenes, medicinal, phenolic, aromatic, chemical, estery, ammonia",
    "Botanical": "plants, floral, herbaceous, grassy, nature, vegetation, water",
    "Industrial": "alcohol, hydrocar, fuel, beverage, coal_tar, gas_station, hydrocarbo"
}

In [9]:
df_1 = df.copy()

In [10]:
# Step 1: Normalize the values to lowercase and replace spaces with underscores
final_cluster_assignment = {
    key: [v.strip().lower().replace(" ", "_") for v in value.split(",")]
    for key, value in final_cluster_assignment.items()
}

# Step 2: Create a reverse mapping from each odor term to its cluster
term_to_cluster = {
    term: cluster
    for cluster, term_list in final_cluster_assignment.items()
    for term in term_list
}

# Step 3: Define the columns to apply the mapping
odor_columns = [f"odor_{i}" for i in range(1, 37)]

# Step 4: Apply the mapping to the DataFrame
for col in odor_columns:
    df_1[col] = df_1[col].map(term_to_cluster)

In [11]:
df_1.to_csv("../data/cluster_descriptors.csv", index = False)

In [12]:
def compute_trend(row, columns=odor_columns, threshold=0.9):
    # Extract values, convert to string, strip whitespace
    values = row[columns].astype(str).str.strip()

    # Remove real NaNs, string "nan", and empty strings
    values = values[~values.isin(['nan', 'NaN', 'NAN', '', 'None'])]

    if len(values) == 0:
        return "Mixed"

    counts = values.value_counts()
    top_label = counts.index[0]
    top_freq = counts.iloc[0]

    if top_freq / len(values) >= threshold:
        return top_label
    else:
        return "Mixed"

In [13]:
df_1["trend_90%"] = df_1.apply(compute_trend, columns = odor_columns, axis = 1, threshold = 0.9)
df_1["trend_80%"] = df_1.apply(compute_trend, columns = odor_columns, axis = 1, threshold = 0.8)
df_1["trend_70%"] = df_1.apply(compute_trend, columns = odor_columns, axis = 1, threshold = 0.7)
df_1["trend_60%"] = df_1.apply(compute_trend, columns = odor_columns, axis = 1, threshold = 0.6)
df_1["trend_50%"] = df_1.apply(compute_trend, columns = odor_columns, axis = 1, threshold = 0.5)
df_1["trend_40%"] = df_1.apply(compute_trend, columns = odor_columns, axis = 1, threshold = 0.4)
df_1["trend_30%"] = df_1.apply(compute_trend, columns = odor_columns, axis = 1, threshold = 0.3)

In [14]:
print(f"Number of non-'Mixed' cells in 'trend_90%': {df_1["trend_90%"].ne("Mixed").sum()} of {len(df_1)}, to a {(100 * df_1["trend_90%"].ne("Mixed").sum() / len(df_1)).round(2)}%")
print(f"Number of non-'Mixed' cells in 'trend_80%': {df_1["trend_80%"].ne("Mixed").sum()} of {len(df_1)}, to a {(100 * df_1["trend_80%"].ne("Mixed").sum() / len(df_1)).round(2)}%")
print(f"Number of non-'Mixed' cells in 'trend_70%': {df_1["trend_70%"].ne("Mixed").sum()} of {len(df_1)}, to a {(100 * df_1["trend_70%"].ne("Mixed").sum() / len(df_1)).round(2)}%")
print(f"Number of non-'Mixed' cells in 'trend_60%': {df_1["trend_60%"].ne("Mixed").sum()} of {len(df_1)}, to a {(100 * df_1["trend_60%"].ne("Mixed").sum() / len(df_1)).round(2)}%")
print(f"Number of non-'Mixed' cells in 'trend_50%': {df_1["trend_50%"].ne("Mixed").sum()} of {len(df_1)}, to a {(100 * df_1["trend_50%"].ne("Mixed").sum() / len(df_1)).round(2)}%")
print(f"Number of non-'Mixed' cells in 'trend_40%': {df_1["trend_40%"].ne("Mixed").sum()} of {len(df_1)}, to a {(100 * df_1["trend_40%"].ne("Mixed").sum() / len(df_1)).round(2)}%")
print(f"Number of non-'Mixed' cells in 'trend_30%': {df_1["trend_30%"].ne("Mixed").sum()} of {len(df_1)}, to a {(100 * df_1["trend_30%"].ne("Mixed").sum() / len(df_1)).round(2)}%")

Number of non-'Mixed' cells in 'trend_90%': 160 of 3424, to a 4.67%
Number of non-'Mixed' cells in 'trend_80%': 166 of 3424, to a 4.85%
Number of non-'Mixed' cells in 'trend_70%': 189 of 3424, to a 5.52%
Number of non-'Mixed' cells in 'trend_60%': 364 of 3424, to a 10.63%
Number of non-'Mixed' cells in 'trend_50%': 729 of 3424, to a 21.29%
Number of non-'Mixed' cells in 'trend_40%': 1183 of 3424, to a 34.55%
Number of non-'Mixed' cells in 'trend_30%': 2056 of 3424, to a 60.05%


In [15]:
df_2 = df_1[['cas_id', 'standardized_smiles', 'trend_30%']]
df_2["trend_30%"].dropna().str.strip().value_counts()

trend_30%
Mixed            1368
Sweet             598
Botanical         592
Woody             295
Food              269
Synthetic          95
Bodily             81
Citrus             57
Industrial         37
Decomposition      27
Waste               5
Name: count, dtype: int64

In [16]:
fingerprints = pd.concat([df_2,
                        pd.DataFrame([list(rdFingerprintGenerator.GetMorganGenerator(radius = 2, fpSize = 1024, includeChirality = True).GetFingerprint(Chem.MolFromSmiles(smiles)).ToBitString()) for smiles in df_2['standardized_smiles']])],
                        axis = 1)

In [17]:
# Execute the reduction of components
data_tsne = fingerprints.iloc[:, 3:]
data_tsne = StandardScaler().fit_transform(data_tsne)
tsne_results_30_1000 = TSNE(random_state = 42, n_components = 2, verbose = 1, perplexity = 30, n_iter = 1000).fit_transform(data_tsne)
tsne_results_30_2000 = TSNE(random_state = 42, n_components = 2, verbose = 1, perplexity = 30, n_iter = 2000).fit_transform(data_tsne)
tsne_results_30_5000 = TSNE(random_state = 42, n_components = 2, verbose = 1, perplexity = 30, n_iter = 5000).fit_transform(data_tsne)
tsne_results_50_1000 = TSNE(random_state = 42, n_components = 2, verbose = 1, perplexity = 50, n_iter = 1000).fit_transform(data_tsne)
tsne_results_50_2000 = TSNE(random_state = 42, n_components = 2, verbose = 1, perplexity = 50, n_iter = 2000).fit_transform(data_tsne)
tsne_results_50_5000 = TSNE(random_state = 42, n_components = 2, verbose = 1, perplexity = 50, n_iter = 5000).fit_transform(data_tsne)

[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 3424 samples in 0.005s...
[t-SNE] Computed neighbors for 3424 samples in 0.134s...
[t-SNE] Computed conditional probabilities for sample 1000 / 3424
[t-SNE] Computed conditional probabilities for sample 2000 / 3424
[t-SNE] Computed conditional probabilities for sample 3000 / 3424
[t-SNE] Computed conditional probabilities for sample 3424 / 3424
[t-SNE] Mean sigma: 8.732696
[t-SNE] KL divergence after 250 iterations with early exaggeration: 91.057503
[t-SNE] KL divergence after 1000 iterations: 2.098150
[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 3424 samples in 0.004s...
[t-SNE] Computed neighbors for 3424 samples in 0.082s...
[t-SNE] Computed conditional probabilities for sample 1000 / 3424
[t-SNE] Computed conditional probabilities for sample 2000 / 3424
[t-SNE] Computed conditional probabilities for sample 3000 / 3424
[t-SNE] Computed conditional probabilities for sample 3424 / 3424
[t-SNE] Mean sigma: 8.732696


In [18]:
labels = fingerprints[['cas_id', 'standardized_smiles', 'trend_30%']].to_numpy()
tsne_results_30_1000 = pd.DataFrame(np.concatenate((labels, tsne_results_30_1000), axis = 1), columns = ['cas_id', 'standardized_smiles', 'trend_30%', 'component1', 'component2'])
tsne_results_30_2000 = pd.DataFrame(np.concatenate((labels, tsne_results_30_2000), axis = 1), columns = ['cas_id', 'standardized_smiles', 'trend_30%', 'component1', 'component2'])
tsne_results_30_5000 = pd.DataFrame(np.concatenate((labels, tsne_results_30_5000), axis = 1), columns = ['cas_id', 'standardized_smiles', 'trend_30%', 'component1', 'component2'])
tsne_results_50_1000 = pd.DataFrame(np.concatenate((labels, tsne_results_50_1000), axis = 1), columns = ['cas_id', 'standardized_smiles', 'trend_30%', 'component1', 'component2'])
tsne_results_50_2000 = pd.DataFrame(np.concatenate((labels, tsne_results_50_2000), axis = 1), columns = ['cas_id', 'standardized_smiles', 'trend_30%', 'component1', 'component2'])
tsne_results_50_5000 = pd.DataFrame(np.concatenate((labels, tsne_results_50_5000), axis = 1), columns = ['cas_id', 'standardized_smiles', 'trend_30%', 'component1', 'component2'])
tsne_results_50_5000.tail(2)

,cas_id,standardized_smiles,trend_30%,component1,component2
3422,288-47-1,c1cscn1,Mixed,-20.368763,-0.340597
3423,38303-23-0,c1nc2c(o1)CCCCCCCCCC2,Mixed,-31.512379,-5.393296


In [19]:
tsne_results_30_1000["trend_30%"].unique()

array(['Mixed', 'Woody', 'Botanical', 'Bodily', 'Food', 'Industrial',
       'Decomposition', 'Sweet', 'Synthetic', 'Citrus', 'Waste'],
      dtype=object)

In [20]:
dfs = [
    ("tsne_results_30_1000", tsne_results_30_1000),
    ("tsne_results_30_2000", tsne_results_30_2000),
    ("tsne_results_30_5000", tsne_results_30_5000),
    ("tsne_results_50_1000", tsne_results_50_1000),
    ("tsne_results_50_2000", tsne_results_50_2000),
    ("tsne_results_50_5000", tsne_results_50_5000),
]

trend_categories = [
    'Mixed', 'Sweet', 'Botanical', 'Woody', 'Food', 'Synthetic', 'Bodily',
    'Citrus', 'Industrial', 'Decomposition', 'Waste'
]

for name, df in dfs:
    df["trend_30%"] = pd.Categorical(df["trend_30%"], categories = trend_categories)
    df = df.sort_values("trend_30%").reset_index(drop = True)
    df.to_csv(f"../data/{name}.csv", index = False)

In [21]:
# Color palette for each odor category
odor_palette = {
    'Mixed': '#999999',
    'Sweet': '#FF69B4',
    'Botanical': '#228B22',
    'Woody': '#8B4513',
    'Food': '#DAA520',
    'Synthetic': '#00BFFF',
    'Bodily': '#800000',
    'Citrus': '#FFA500',
    'Industrial': '#4682B4',
    'Decomposition': '#808000',
    'Waste': '#556B2F',
}

# Which categories to highlight in each subplot
subplot_highlights = [
    list(odor_palette.keys()),                      # Subplot 0: All
    ['Mixed', 'Sweet', 'Bodily'],                   # Subplot 1
    ['Mixed', 'Botanical', 'Citrus'],               # Subplot 2
    ['Mixed', 'Woody', 'Industrial'],               # Subplot 3
    ['Mixed', 'Food', 'Decomposition'],             # Subplot 4
    ['Mixed', 'Synthetic', 'Waste'],                # Subplot 5
]

# Titles for each subplot
subplot_titles = [
    "All Categories",
    "Sweet & Bodily",
    "Botanical & Citrus",
    "Woody & Industrial",
    "Food & Decomposition",
    "Synthetic & Waste"
]

In [32]:
def create_tsne_trend_subplots_adjusted(df: pd.DataFrame, name: str, category_col: str = "trend_30%"):
    df[category_col] = df[category_col].astype(str)
    
    # Calculate axis limits with margin
    margin_ratio = 0.02  # 2% margin

    x_min, x_max = df["component1"].min(), df["component1"].max()
    y_min, y_max = df["component2"].min(), df["component2"].max()

    x_range = x_max - x_min
    y_range = y_max - y_min

    x_limits = [x_min - margin_ratio * x_range, x_max + margin_ratio * x_range]
    y_limits = [y_min - margin_ratio * y_range, y_max + margin_ratio * y_range]

    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=subplot_titles,
        horizontal_spacing=0.05,
        vertical_spacing=0.1
    )

    for i, highlight_categories in enumerate(subplot_highlights):
        row = i // 3 + 1
        col = i % 3 + 1

        df_filtered = df[df[category_col].isin(highlight_categories)].copy()
        df_filtered["sort_key"] = df_filtered[category_col].apply(lambda x: 0 if x == "Mixed" else 1)
        df_filtered = df_filtered.sort_values("sort_key")

        for label in highlight_categories:
            df_subset = df_filtered[df_filtered[category_col] == label]
            fig.add_trace(
                go.Scatter(
                    x=df_subset["component1"],
                    y=df_subset["component2"],
                    mode='markers',
                    marker=dict(
                        size=6,
                        color=odor_palette.get(label, "#000000"),
                        opacity=0.6 if label == "Mixed" else 1.0,
                    ),
                    name=label if i == 0 else "",
                    showlegend=(i == 0)
                ),
                row=row,
                col=col
            )

    fig.update_layout(
        height=900,
        width=1500,
        title_text=f"tSNE Trend Subplots: {name}",
        plot_bgcolor='white',
        paper_bgcolor='white',
        font=dict(size=14),

        # Legend at bottom center
        legend=dict(
            orientation="h",   # horizontal legend
            yanchor="top",
            y=-0.1,           # slightly below the plot
            xanchor="center",
            x=0.5,
            bgcolor="rgba(255,255,255,0)",  # transparent background
            bordercolor="Black",
            borderwidth=0,
            font=dict(size=22)
        ),
        margin=dict(b=120)  # bottom margin to give space for legend
    )

    for i in range(1, 7):
        fig.update_xaxes(
            title_text="tSNE dimension 1",
            range=x_limits,
            row=(i-1)//3+1,
            col=(i-1)%3+1,
            showgrid=False,
            showline=True,
            linecolor='black',
            mirror=True,
            tickfont=dict(size=12)
        )
        fig.update_yaxes(
            title_text="tSNE dimension 2",
            range=y_limits,
            row=(i-1)//3+1,
            col=(i-1)%3+1,
            showgrid=False,
            showline=True,
            linecolor='black',
            mirror=True,
            tickfont=dict(size=12)
        )

    output_path = f"../figures/tsne_trend_subplots_{name}_adjusted.png"
    pio.write_image(fig, output_path, format="png", width=1500, height=900, scale=3)
    print(f"Saved: {output_path}")

In [33]:
for name, df in dfs:
    create_tsne_trend_subplots_adjusted(df, name)

Saved: ../figures/tsne_trend_subplots_tsne_results_30_1000_adjusted.png
Saved: ../figures/tsne_trend_subplots_tsne_results_30_2000_adjusted.png
Saved: ../figures/tsne_trend_subplots_tsne_results_30_5000_adjusted.png
Saved: ../figures/tsne_trend_subplots_tsne_results_50_1000_adjusted.png
Saved: ../figures/tsne_trend_subplots_tsne_results_50_2000_adjusted.png
Saved: ../figures/tsne_trend_subplots_tsne_results_50_5000_adjusted.png


In [24]:
def smiles_embedding(smiles_list, model_name, max_lenght, padding = True, trust_remote_code = True):
    """
    Featurizes a list of SMILES strings using pretrained large larguage models  

    Args:
        smiles_list : list of str
            A list of SMILES strings representing molecular structures.
        model_name : str
            The name of the pre-trained model to use for featurization.
        max_length : int, optional (default=512)
            The maximum length of the tokenized input. Inputs longer than this will be truncate
    padding : bool, optional (default=True)
        Whether to pad the tokenized inputs to the same length. If True, the tokenizer will
        pad the inputs to the maximum sequence length in the batch. 
    trust_remote_code : bool, optional (default=False)
            Whether to trust custom code in the model repository (required for MolFormer).

    Returns:
    --------
    numpy.ndarray
        A 2D NumPy array of shape (number_of_smiles, max_lenth), where each row corresponds to
        the embedding of a SMILES string from the input list.
    """
    
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code = trust_remote_code)
    chemberta = AutoModelForMaskedLM.from_pretrained(model_name, trust_remote_code = trust_remote_code)
    
    embeddings_cls = torch.zeros(len(smiles_list), max_lenght)
    
    with torch.no_grad():
        for i, smiles in enumerate(tqdm(smiles_list)):
            encoded_input = tokenizer(smiles, return_tensors = "pt", padding = padding, truncation = True)
            model_output = chemberta(**encoded_input)
            
            embeddings_cls[i] = model_output[0][::, 0, ::]
            
    return pd.DataFrame(embeddings_cls.numpy(), columns = [f"feature_{i}" for i in range(embeddings_cls.shape[1])])

In [25]:
molformer_embedding = pd.concat([df_2,
                                smiles_embedding(df_2["standardized_smiles"], 'ibm/MolFormer-XL-both-10pct', 2362)],
                                axis = 1)

  0%|          | 0/3424 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [26]:
molformer_embedding.head(2)

,cas_id,standardized_smiles,trend_30%,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,...,feature_2352,feature_2353,feature_2354,feature_2355,feature_2356,feature_2357,feature_2358,feature_2359,feature_2360,feature_2361
0,103-64-0,BrC=Cc1ccccc1,Mixed,-10.093344,-10.337275,-10.321631,-10.287940,0.786431,0.278108,-3.062706,...,-9.663470,-10.010611,-10.060002,-9.752295,-10.102135,-9.932281,-9.389063,-9.270222,-9.666122,-10.219555
1,29171-20-8,C#CC(C)(O)CCC=C(C)C,Woody,-11.430017,-11.426505,-11.345511,-11.069971,5.130373,-2.085578,3.002152,...,-11.185769,-10.118696,-11.184937,-10.716600,-10.835524,-10.435621,-11.208942,-10.442491,-10.720177,-11.283046


In [27]:
# Execute the reduction of components
data_tsne = molformer_embedding.iloc[:, 3:]
data_tsne = StandardScaler().fit_transform(data_tsne)
molformer_tsne_results_30_1000 = TSNE(random_state = 42, n_components = 2, verbose = 1, perplexity = 30, n_iter = 1000).fit_transform(data_tsne)
molformer_tsne_results_30_2000 = TSNE(random_state = 42, n_components = 2, verbose = 1, perplexity = 30, n_iter = 2000).fit_transform(data_tsne)
molformer_tsne_results_30_5000 = TSNE(random_state = 42, n_components = 2, verbose = 1, perplexity = 30, n_iter = 5000).fit_transform(data_tsne)
molformer_tsne_results_50_1000 = TSNE(random_state = 42, n_components = 2, verbose = 1, perplexity = 50, n_iter = 1000).fit_transform(data_tsne)
molformer_tsne_results_50_2000 = TSNE(random_state = 42, n_components = 2, verbose = 1, perplexity = 50, n_iter = 2000).fit_transform(data_tsne)
molformer_tsne_results_50_5000 = TSNE(random_state = 42, n_components = 2, verbose = 1, perplexity = 50, n_iter = 5000).fit_transform(data_tsne)

[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 3424 samples in 0.008s...
[t-SNE] Computed neighbors for 3424 samples in 0.330s...
[t-SNE] Computed conditional probabilities for sample 1000 / 3424
[t-SNE] Computed conditional probabilities for sample 2000 / 3424
[t-SNE] Computed conditional probabilities for sample 3000 / 3424
[t-SNE] Computed conditional probabilities for sample 3424 / 3424
[t-SNE] Mean sigma: 8.506760
[t-SNE] KL divergence after 250 iterations with early exaggeration: 75.199356
[t-SNE] KL divergence after 1000 iterations: 1.366584
[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 3424 samples in 0.008s...
[t-SNE] Computed neighbors for 3424 samples in 0.303s...
[t-SNE] Computed conditional probabilities for sample 1000 / 3424
[t-SNE] Computed conditional probabilities for sample 2000 / 3424
[t-SNE] Computed conditional probabilities for sample 3000 / 3424
[t-SNE] Computed conditional probabilities for sample 3424 / 3424
[t-SNE] Mean sigma: 8.506760


In [28]:
labels = molformer_embedding[['cas_id', 'standardized_smiles', 'trend_30%']].to_numpy()
molformer_tsne_results_30_1000 = pd.DataFrame(np.concatenate((labels, molformer_tsne_results_30_1000), axis = 1), columns = ['cas_id', 'standardized_smiles', 'trend_30%', 'component1', 'component2'])
molformer_tsne_results_30_2000 = pd.DataFrame(np.concatenate((labels, molformer_tsne_results_30_2000), axis = 1), columns = ['cas_id', 'standardized_smiles', 'trend_30%', 'component1', 'component2'])
molformer_tsne_results_30_5000 = pd.DataFrame(np.concatenate((labels, molformer_tsne_results_30_5000), axis = 1), columns = ['cas_id', 'standardized_smiles', 'trend_30%', 'component1', 'component2'])
molformer_tsne_results_50_1000 = pd.DataFrame(np.concatenate((labels, molformer_tsne_results_50_1000), axis = 1), columns = ['cas_id', 'standardized_smiles', 'trend_30%', 'component1', 'component2'])
molformer_tsne_results_50_2000 = pd.DataFrame(np.concatenate((labels, molformer_tsne_results_50_2000), axis = 1), columns = ['cas_id', 'standardized_smiles', 'trend_30%', 'component1', 'component2'])
molformer_tsne_results_50_5000 = pd.DataFrame(np.concatenate((labels, molformer_tsne_results_50_5000), axis = 1), columns = ['cas_id', 'standardized_smiles', 'trend_30%', 'component1', 'component2'])
molformer_tsne_results_50_5000.tail(2)

,cas_id,standardized_smiles,trend_30%,component1,component2
3422,288-47-1,c1cscn1,Mixed,74.692535,16.236267
3423,38303-23-0,c1nc2c(o1)CCCCCCCCCC2,Mixed,-36.519222,-10.471739


In [29]:
molformer_dfs = [
    ("molformer_tsne_results_30_1000", molformer_tsne_results_30_1000),
    ("molformer_tsne_results_30_2000", molformer_tsne_results_30_2000),
    ("molformer_tsne_results_30_5000", molformer_tsne_results_30_5000),
    ("molformer_tsne_results_50_1000", molformer_tsne_results_50_1000),
    ("molformer_tsne_results_50_2000", molformer_tsne_results_50_2000),
    ("molformer_tsne_results_50_5000", molformer_tsne_results_50_5000),
]

trend_categories = [
    'Mixed', 'Sweet', 'Botanical', 'Woody', 'Food', 'Synthetic', 'Bodily',
    'Citrus', 'Industrial', 'Decomposition', 'Waste'
]

for name, df in molformer_dfs:
    df["trend_30%"] = pd.Categorical(df["trend_30%"], categories = trend_categories)
    df = df.sort_values("trend_30%").reset_index(drop = True)
    df.to_csv(f"../data/{name}.csv", index = False)

In [34]:
for name, df in molformer_dfs:
    create_tsne_trend_subplots_adjusted(df, name)

Saved: ../figures/tsne_trend_subplots_molformer_tsne_results_30_1000_adjusted.png
Saved: ../figures/tsne_trend_subplots_molformer_tsne_results_30_2000_adjusted.png
Saved: ../figures/tsne_trend_subplots_molformer_tsne_results_30_5000_adjusted.png
Saved: ../figures/tsne_trend_subplots_molformer_tsne_results_50_1000_adjusted.png
Saved: ../figures/tsne_trend_subplots_molformer_tsne_results_50_2000_adjusted.png
Saved: ../figures/tsne_trend_subplots_molformer_tsne_results_50_5000_adjusted.png


In [31]:
fingerprints = pd.concat([df_2,
                        pd.DataFrame([list(rdFingerprintGenerator.GetMorganGenerator(radius = 2, fpSize = 1024, includeChirality = True).GetFingerprint(Chem.MolFromSmiles(smiles)).ToBitString()) for smiles in df_2['standardized_smiles']])],
                        axis = 1)